In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [17]:
clean_file_path = 'codebar-AI-fundamentals-pixels-banking-ML-model/data/pixels_banking_loan_repayment_clean.csv'
df_model_ready = pd.read_csv(clean_file_path)

In [18]:
X = df_model_ready.drop(columns=['loan_repaid'])
y = df_model_ready['loan_repaid']

# Split into Training and Testing sets (20% test size)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('Training features shape:', X_train.shape)
print('Testing features shape:', X_test.shape)

Training features shape: (960, 39)
Testing features shape: (240, 39)


In [19]:
# List the columns that need to be scaled down
numeric_cols_to_scale = [
    'age', 'tenure_months', 'monthly_income', 'avg_monthly_balance', 
    'credit_score', 'num_products', 'customer_service_calls', 
    'missed_payments_6m', 'support_rating', 'monthly_fees', 
    'digital_engagement_score', 'loan_amount', 'loan_term_months', 
    'debt_to_income_ratio', 'interest_rate'
]

# Initialize the scaler
scaler = StandardScaler()

# Create copies so we don't disrupt our original split dataframes
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Secure Rule: fit_transform training data, only transform test data
X_train_scaled[numeric_cols_to_scale] = scaler.fit_transform(X_train[numeric_cols_to_scale])
X_test_scaled[numeric_cols_to_scale] = scaler.transform(X_test[numeric_cols_to_scale])

print("Features successfully scaled!")

Features successfully scaled!


In [20]:
model = LogisticRegression(class_weight={0: 5, 1: 1}, max_iter=1000, solver='liblinear')

In [21]:
model.fit(X_train_scaled, y_train)
print('Model trained successfully on scaled data')

Model trained successfully on scaled data


In [22]:
# get raw probabilities for Class 1 (Repaid) using the trained model
probabilities = model.predict_proba(X_test_scaled)[:, 1]

# apply the strict 60% threshold 
custom_threshold = 0.6
y_pred_strict = (probabilities >= custom_threshold).astype(int)

In [23]:
results = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred_strict
})

results.head(10)

,Actual,Predicted
0,0,0
1,1,0
2,0,0
3,0,0
4,0,0
5,1,0
6,0,0
7,1,1
8,1,1
9,0,0


In [24]:
accuracy_strict = accuracy_score(y_test, y_pred_strict)

print('Accuracy:', round(accuracy_strict, 3))
print('Accuracy percentage:', round(accuracy_strict * 100, 1), '%')

Accuracy: 0.504
Accuracy percentage: 50.4 %


In [25]:
cm = confusion_matrix(y_test, y_pred_strict)

cm_df = pd.DataFrame(
    cm,
    index=['Actual Not Repaid (0)', 'Actual Repaid (1)'],
    columns=['Predicted Not Repaid (0)', 'Predicted Repaid (1)']
)

cm_df

,Predicted Not Repaid (0),Predicted Repaid (1)
Actual Not Repaid (0),66,8
Actual Repaid (1),111,55


In [26]:
print("\n--- Corrected Confusion Matrix ---")
print(cm_df)


--- Corrected Confusion Matrix ---
                       Predicted Not Repaid (0)  Predicted Repaid (1)
Actual Not Repaid (0)                        66                     8
Actual Repaid (1)                           111                    55


In [27]:
print("\n--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred_strict))


--- Detailed Classification Report ---
              precision    recall  f1-score   support

           0       0.37      0.89      0.53        74
           1       0.87      0.33      0.48       166

    accuracy                           0.50       240
   macro avg       0.62      0.61      0.50       240
weighted avg       0.72      0.50      0.49       240



In [28]:
coefficients = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

print(coefficients)

                            Feature  Coefficient
35          uses_mobile_app_Unknown     1.117324
28            account_type_Everyday     0.749409
4                      credit_score     0.711756
15         credit_score_was_missing     0.624854
19                region_North West     0.603495
24                   region_Unknown     0.524100
36              uses_mobile_app_Yes     0.459233
29                account_type_Plus     0.453636
17                    region_London     0.345030
16       monthly_income_was_missing     0.294520
2                    monthly_income     0.267518
9                      monthly_fees     0.258385
1                     tenure_months     0.207333
23                region_South West     0.161855
14                    interest_rate     0.057720
5                      num_products     0.037931
33              has_credit_card_Yes     0.018413
10         digital_engagement_score    -0.009863
20          region_Northern Ireland    -0.018554
12                 l